# ATP Data Merging: GitHub + Betting Odds

This notebook merges two ATP tennis match datasets:
1. **GitHub dataset**: Detailed match statistics (aces, double faults, serve %, etc.)
2. **Betting dataset**: Betting odds from various bookmakers

## The Challenge
These datasets don't share a common match ID, so we need to match records using:
- Year
- Tournament name (may differ: "Roland Garros" vs "French Open")
- Player names (format differs: "Roger Federer" vs "Federer R.")
- Round

We'll use a combination of explicit mappings and fuzzy matching.

In [1]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR = Path("../../raw_data/")
OUTPUT_DIR = Path("../data/")
YEARS = list(range(2015, 2024))

print(f"Processing years: {YEARS[0]} - {YEARS[-1]}")

Processing years: 2015 - 2023


In [2]:
# Load GitHub data
github_dfs = []
for year in YEARS:
    file_path = INPUT_DIR / f"github/atp_matches_{year}.csv"
    df_year = pd.read_csv(file_path)
    df_year["year"] = year
    github_dfs.append(df_year)
    print(f"GitHub {year}: {len(df_year)} matches")

df_github = pd.concat(github_dfs, ignore_index=True)
print(f"\nTotal GitHub matches: {len(df_github)}")

GitHub 2015: 2943 matches
GitHub 2016: 2941 matches
GitHub 2017: 2911 matches
GitHub 2018: 2897 matches
GitHub 2019: 2806 matches
GitHub 2020: 1462 matches
GitHub 2021: 2733 matches
GitHub 2022: 2917 matches
GitHub 2023: 2986 matches

Total GitHub matches: 24596


In [3]:
# Load Betting data
bets_dfs = []
for year in YEARS:
    file_path = INPUT_DIR / f"bets/{year}.xlsx"
    df_year = pd.read_excel(file_path)
    if year == 2018:
        df_year = df_year.drop(columns=["EXW", "EXL", "LBW", "LBL"])

    df_year["year"] = year
    bets_dfs.append(df_year)
    print(f"Bets {year}: {len(df_year)} matches")

df_bets = pd.concat(bets_dfs, ignore_index=True)
print(f"\nTotal Bets matches: {len(df_bets)}")

Bets 2015: 2630 matches
Bets 2016: 2626 matches
Bets 2017: 2633 matches
Bets 2018: 2637 matches
Bets 2019: 2610 matches
Bets 2020: 1267 matches
Bets 2021: 2489 matches
Bets 2022: 2632 matches
Bets 2023: 2703 matches

Total Bets matches: 22227


In [4]:
print("GitHub columns:")
print(df_github.columns.tolist())
print(f"\nBets columns:")
print(df_bets.columns.tolist())

GitHub columns:
['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points', 'year']

Bets columns:
['ATP', 'Location', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank', 'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'EXW', 'EXL', 'LBW', 'LBL', 'PSW', 'PSL', 'MaxW', 'MaxL', 'AvgW', 'AvgL', 'y

In [5]:
df_github.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,year
0,2015-339,Brisbane,Hard,28,A,20150104,1,105357,NaN,WC,...,20.0,5.0,8.0,1.0,5.0,153.0,328.0,220.0,221.0,2015
1,2015-339,Brisbane,Hard,28,A,20150104,2,103813,NaN,NaN,...,26.0,19.0,13.0,3.0,8.0,73.0,689.0,123.0,440.0,2015
2,2015-339,Brisbane,Hard,28,A,20150104,3,105902,NaN,WC,...,22.0,5.0,8.0,10.0,15.0,125.0,430.0,21.0,1730.0,2015
3,2015-339,Brisbane,Hard,28,A,20150104,4,104871,NaN,NaN,...,30.0,8.0,10.0,1.0,3.0,31.0,1195.0,72.0,691.0,2015
4,2015-339,Brisbane,Hard,28,A,20150104,5,105373,NaN,NaN,...,40.0,19.0,15.0,4.0,8.0,34.0,1094.0,110.0,505.0,2015


In [6]:
df_bets.head()

,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,EXL,LBW,LBL,PSW,PSL,MaxW,MaxL,AvgW,AvgL,year
0,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Duckworth J.,...,1.20,4.33,1.20,4.67,1.23,4.73,1.23,4.31,1.20,2015
1,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Kokkinakis T.,...,1.48,2.62,1.44,2.67,1.53,2.80,1.53,2.62,1.47,2015
2,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Chardy J.,...,3.20,1.33,3.25,1.34,3.50,1.36,3.50,1.32,3.30,2015
3,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Tomic B.,...,2.30,1.62,2.20,1.65,2.37,1.67,2.37,1.61,2.25,2015
4,1,Brisbane,Brisbane International,2015-01-06,ATP250,Outdoor,Hard,1st Round,3.0,Kukushkin M.,...,2.50,1.44,2.62,1.48,2.84,1.58,2.84,1.50,2.53,2015


## We need to match tournaments between the df's, we will match by df_github: tourney_name -> df_bets: Location

### First we remove the tournaments that do not follow standard knock out draw format and fix trailing spaces

In [7]:
df_github=df_github[~df_github['tourney_name'].str.contains('Davis Cup', na=False)]
df_github = df_github[~df_github['tourney_name'].isin(['United Cup', 'Tour Finals', 'NextGen Finals','Rio Olympics', 'Tokyo Olympics', 'Atp Cup', 'Laver Cup'])]
df_github['tourney_name'] = df_github['tourney_name'].str.rstrip()
df_github.rename(columns={'tourney_name': 'tourney_location'}, inplace=True)
df_github['tourney_location'] = df_github['tourney_location'].replace({"ATP Rio de Janeiro": "Rio de Janeiro"})

df_bets = df_bets[~df_bets['Tournament'].isin(['United Cup', 'Tour Finals', 'Masters Cup', 'NextGen Finals','Tokyo Olympics','Atp Cup','Laver Cup'])]

### For some tournaments in df_github column 'tourney_name' does not represent the location so we will have to change that manually

In [8]:
df_github['tourney_location'] = df_github['tourney_location'].replace({"Australian Open": "Melbourne", "Roland Garros": "Paris", "Queen's Club": "Queens Club", "Wimbledon": "London", "Us Open": "New York", "US Open": "New York", "Astana": "Nur-Sultan", "Paris Masters": "Paris 2"})

### Canada Masters event in odd years is played in Montreal and in even years in Toronto up until 2020 and then it was reversed

In [9]:
mask = df_github['tourney_location'] == 'Canada Masters'

df_github.loc[mask & (df_github['year'] <= 2020) & (df_github['year'] % 2 == 1), 'tourney_location'] = 'Montreal'
df_github.loc[mask & (df_github['year'] <= 2020) & (df_github['year'] % 2 == 0), 'tourney_location'] = 'Toronto'
df_github.loc[mask & (df_github['year'] > 2020) & (df_github['year'] % 2 == 1), 'tourney_location'] = 'Toronto'
df_github.loc[mask & (df_github['year'] > 2020) & (df_github['year'] % 2 == 0), 'tourney_location'] = 'Montreal'

### Some small fixes df_bets tournament locations

In [10]:
df_bets.loc[df_bets["Tournament"] == "Western & Southern Financial Group Masters", "Location"] = "Cincinnati"

auckland_mask = (df_bets['year'] == 2020) & (df_bets['Location'] == 'Auckland') & (df_bets['Tournament'] == 'ASB Classic')
adelaide_mask = (df_bets['year'] == 2020) & (df_bets['Location'] == 'Adelaide') & (df_bets['Tournament'] == 'Adelaide International')

df_bets.loc[auckland_mask, ['Location', 'Tournament']] = ['Adelaide', 'Adelaide International']
df_bets.loc[adelaide_mask, ['Location', 'Tournament']] = ['Auckland', 'ASB Classic']

### Let's see what the overlap is now

In [11]:
def compare_overlap(df1, col1, df2, col2):
    s1 = set(df1[col1].unique())
    s2 = set(df2[col2].unique())

    only_1 = s1 - s2
    only_2 = s2 - s1
    both = s1 & s2

    print(f"Unique in df1[{col1}]: {len(s1)}")
    print(f"Unique in df2[{col2}]: {len(s2)}")
    print(f"Overlap: {len(both)}")

    print(f"\nIn df1[{col1}] but NOT in df2[{col2}] ({len(only_1)}):")
    for v in sorted(only_1):
        print(v)

    print(f"\nIn df2[{col2}] but NOT in df1[{col1}] ({len(only_2)}):")
    for v in sorted(only_2):
        print(v)

    return {
        "only_df1": sorted(only_1),
        "only_df2": sorted(only_2),
        "overlap": sorted(both),
    }

res = compare_overlap(df_github, 'tourney_location', df_bets, 'Location')

Unique in df1[tourney_location]: 105
Unique in df2[Location]: 96
Overlap: 84

In df1[tourney_location] but NOT in df2[Location] (21):
Adelaide 1
Adelaide 2
Belgrade 2
Cincinnati Masters
Cologne 1
Cologne 2
Dubai
Great Ocean Road Open
Indian Wells Masters
Madrid Masters
Miami Masters
Monte Carlo Masters
Murray River Open
Naples
Paris 2
Rio De Janeiro
Rome Masters
Shanghai Masters
Shenzhen
St Petersburg
s Hertogenbosch

In df2[Location] but NOT in df1[tourney_location] (12):
's-Hertogenbosch
Cincinnati
Cologne
Dubai 
Indian Wells
Madrid
Miami
Monte Carlo
Napoli
Rome
Shanghai
Shenzhen 


### When there's more than one tournament in a given location we need to add numbers to 'Location' in bets_df

In [12]:
df_bets.loc[df_bets["Tournament"] == "Adelaide International 1", "Location"] = "Adelaide 1"
df_bets.loc[df_bets["Tournament"] == "Adelaide International 2", "Location"] = "Adelaide 2"

df_bets.loc[df_bets["Tournament"] == "bett1HULKS Indoors", "Location"] = "Cologne 1"
df_bets.loc[df_bets["Tournament"] == "bett1HULKS Championship", "Location"] = "Cologne 2"

df_bets.loc[df_bets['Tournament']== 'BNP Paribas Masters', 'Location']='Paris 2'
df_bets.loc[df_bets['Tournament']== 'Belgrade Open', 'Location']='Belgrade 2'

In [13]:
res = compare_overlap(df_github, 'tourney_location', df_bets, 'Location')

Unique in df1[tourney_location]: 105
Unique in df2[Location]: 101
Overlap: 90

In df1[tourney_location] but NOT in df2[Location] (15):
Cincinnati Masters
Dubai
Great Ocean Road Open
Indian Wells Masters
Madrid Masters
Miami Masters
Monte Carlo Masters
Murray River Open
Naples
Rio De Janeiro
Rome Masters
Shanghai Masters
Shenzhen
St Petersburg
s Hertogenbosch

In df2[Location] but NOT in df1[tourney_location] (11):
's-Hertogenbosch
Cincinnati
Dubai 
Indian Wells
Madrid
Miami
Monte Carlo
Napoli
Rome
Shanghai
Shenzhen 


In [14]:
from rapidfuzz import process, fuzz

def fuzzy_suggest_unmatched(unmatched_values, candidates, *, cutoff):
    candidates = list(candidates)

    out = {}
    for v in unmatched_values:
        matches = process.extract(
            v,
            candidates,
            scorer=fuzz.WRatio,
        )

        # keep only matches above cutoff
        out[v] = [(m, score) for m, score, _ in matches if score >= cutoff]

    return out


In [15]:
suggestions = fuzzy_suggest_unmatched(
    res["only_df1"],
    df_bets["Location"].unique(),
    cutoff=75
)

In [16]:
suggestions

{'Cincinnati Masters': [('Cincinnati', 90.0)],
 'Dubai': [('Dubai ', 95.0)],
 'Great Ocean Road Open': [],
 'Indian Wells Masters': [('Indian Wells', 90.0)],
 'Madrid Masters': [('Madrid', 90.0)],
 'Miami Masters': [('Miami', 90.0)],
 'Monte Carlo Masters': [('Monte Carlo', 90.0)],
 'Murray River Open': [],
 'Naples': [],
 'Rio De Janeiro': [('Rio de Janeiro', 92.85714285714286)],
 'Rome Masters': [('Rome', 90.0)],
 'Shanghai Masters': [('Shanghai', 90.0)],
 'Shenzhen': [('Shenzhen ', 95.0)],
 'St Petersburg': [('St. Petersburg', 96.2962962962963)],
 's Hertogenbosch': [("'s-Hertogenbosch", 90.32258064516128)]}

In [17]:
rename_map = {
    src: matches[0][0]
    for src, matches in suggestions.items()
    if len(matches) > 0
}

df_github['tourney_location'] = df_github['tourney_location'].replace(rename_map)

In [18]:
df_github['tourney_location'] = df_github['tourney_location'].replace({'Great Ocean Road Open': 'Melbourne 2', 'Murray River Open': 'Melbourne 3'})
df_github.loc[(df_github['draw_size'] == 32) & (df_github['tourney_location'] == 'Melbourne'), 'tourney_location'] = 'Melbourne 4'

df_bets["Location"] = df_bets["Location"].replace({'Napoli':'Naples'})
df_bets.loc[df_bets['Tournament']== 'Great Ocean Road Open', 'Location']='Melbourne 2'
df_bets.loc[df_bets['Tournament']== 'Murray River Open', 'Location']='Melbourne 3'
df_bets.loc[df_bets['Tournament']== 'Melbourne Summer Set', 'Location']='Melbourne 4'

In [19]:
res = compare_overlap(df_github, 'tourney_location', df_bets, 'Location')


Unique in df1[tourney_location]: 104
Unique in df2[Location]: 104
Overlap: 104

In df1[tourney_location] but NOT in df2[Location] (0):

In df2[Location] but NOT in df1[tourney_location] (0):


## Now we match player names, df_github has full names while df_bets last name and initial/s

In [20]:
def transform_name(name):
    name_parts = name.split()
    first_name = name_parts[0]
    last_name = ' '.join(name_parts[1:])
    return f"{last_name} {first_name[0]}."

df_github['shortened_winner_name']=df_github['winner_name'].apply(transform_name)
df_github['shortened_loser_name']=df_github['loser_name'].apply(transform_name)

In [21]:
def check_one_to_one_loser_mapping(df):
    bad_name_to_id = (
        df.groupby('shortened_loser_name')['loser_id']
        .nunique()
        .gt(1)
    )

    bad_id_to_name = (
        df.groupby('loser_id')['shortened_loser_name']
        .nunique()
        .gt(1)
    )

    if bad_name_to_id.any() or bad_id_to_name.any():
        print("One-to-one mapping violated")

        if bad_name_to_id.any():
            print("\nshortened_loser_name → multiple loser_id:")
            print(bad_name_to_id[bad_name_to_id].index.tolist())

        if bad_id_to_name.any():
            print("\nloser_id → multiple shortened_loser_name:")
            print(bad_id_to_name[bad_id_to_name].index.tolist())
    else:
        print("One-to-one mapping OK")

check_one_to_one_loser_mapping(df_github)


One-to-one mapping violated

shortened_loser_name → multiple loser_id:
['Kuznetsov A.', 'Landaluce M.', 'Martin A.', 'Nava E.', 'Zhang Z.']


### We discovered some factual issues with the data that needed correction firstly

In [22]:
df_github.loc[df_github['year'] == 2023, 'loser_name'] = df_github.loc[df_github['year'] == 2023, 'loser_name'].replace('Eduardo Nava', 'Emilio Nava')
df_github.loc[df_github['year'] == 2023, 'loser_id'] = df_github.loc[df_github['year'] == 2023, 'loser_id'].replace(124013, 207182)
df_github.loc[(df_github['year'] == 2023) & (df_github['loser_id'] == 212021), 'loser_hand'] = 'R'
df_github.loc[(df_github['year'] == 2023) & (df_github['loser_id'] == 212021), 'loser_ht'] = 191
df_github.loc[(df_github['year'] == 2023) & (df_github['loser_id'] == 212021), 'loser_id'] = 211776


In [23]:
df_bets.loc[(df_bets['year'] == 2023) & (df_bets['Tournament'] == 'Atlanta Open') & (df_bets['Winner'] == 'Martin A.'), 'Winner'] = 'Martin An.'
df_bets.loc[(df_bets['year'] == 2023) & (df_bets['Tournament'] == 'Atlanta Open') & (df_bets['Loser'] == 'Martin A.'), 'Loser'] = 'Martin An.'
df_bets.loc[(df_bets['year'] == 2022) & (df_bets['Tournament'] == 'Atlanta Open') & (df_bets['Winner'] == 'Martin A.'), 'Winner'] = 'Martin An.'
df_bets.loc[(df_bets['year'] == 2022) & (df_bets['Tournament'] == 'Atlanta Open') & (df_bets['Loser'] == 'Martin A.'), 'Loser'] = 'Martin An.'
df_bets.loc[(df_bets['year'] == 2021) & (df_bets['Location'] == 'Winston-Salem') & (df_bets['Loser'] == 'Nava E.'), 'Loser'] = 'Nava Ed.'

### Differentiate the players

In [24]:
df_github.loc[df_github['loser_id'] == 111190, 'shortened_loser_name'] = 'Zhang Zh.'
df_github.loc[df_github['winner_id'] == 111190, 'shortened_winner_name'] = 'Zhang Zh.'
df_github.loc[df_github['loser_id'] == 105585, 'shortened_loser_name'] = 'Zhang Ze.'
df_github.loc[df_github['winner_id'] == 105585, 'shortened_winner_name'] = 'Zhang Ze.'
df_github.loc[df_github['loser_id'] == 104864, 'shortened_loser_name'] = 'Kuznetsov Al.'
df_github.loc[df_github['winner_id'] == 104864, 'shortened_winner_name'] = 'Kuznetsov Al.'
df_github.loc[df_github['loser_id'] == 105723, 'shortened_loser_name'] = 'Kuznetsov A.'
df_github.loc[df_github['winner_id'] == 105723, 'shortened_winner_name'] = 'Kuznetsov A.'
df_github.loc[df_github['loser_id'] == 124013, 'shortened_loser_name'] = 'Nava Ed.'
df_github.loc[df_github['winner_id'] == 124013, 'shortened_winner_name'] = 'Nava Ed.'
df_github.loc[df_github['loser_id'] == 211346, 'shortened_loser_name'] = 'Martin An.'
df_github.loc[df_github['winner_id'] == 211346, 'shortened_winner_name'] = 'Martin An.'

In [25]:
check_one_to_one_loser_mapping(df_github)


One-to-one mapping OK


### Some inconsistencies in bets_df player names

In [26]:
def replace_names(df):
    replacements = {
        "Zhang Z.": "Zhang Ze",
        "Herbert P.H": "Herbert P.H.",
        "Bautista Agut R.": "Bautista R.",
        "Zayid M. S.": "Zayid M.S.",
        "Munoz De La Nava D.": "Munoz-De La Nava D.",
        "Carreno Busta P.": "Carreno-Busta P.",
        "Del Potro J. M.": "Del Potro J.M.",
        "Silva F.F.": "Silva F.",
        "Dolgopolov A.":"Dolgopolov O.",
        "Aragone JC": "Aragone J.C.",
        "Aragone J.": "Aragone J.C.",
        "Hernandez-Fernandez J": "Hernandez-Fernandez J.",
        "Bu Y.": "Yunchaokete B.",
        "Varillas J. P.": "Varillas J.P.",
        "Tseng C. H.": "Tseng C.H.",
        "O Connell C.": "O'Connell C.",
        "Zayid M.": "Zayid M.S.",
        "Al Mutawa J.": "Ali Mutawa J.M.",
        "Barrios M.": "Barrios Vera M.T.",
        "Galan D.": "Galan D.E.",
        "Meligeni Alves F.": "Meligeni Rodrigues F",
        "Li Zh.": "Li Z.",
        "Lu Y.H.": "Lu Y.",
        "Moroni G.": "Moroni G.M.",
        "Monteiro J.": "Monteiro T.",
        "Silva F.": "Ferreira Silva F.",
        "Tyurnev E.": "Tiurnev E.",
        "Zhang Ze.": "Zhang Ze",
    }

    for old_name, new_name in replacements.items():
        df['Loser'] = df['Loser'].replace({old_name: new_name})
        df['Winner']= df['Winner'].replace({old_name:new_name})

replace_names(df_bets)

### There is also an issue with two matches where two matches have incorrect winner/loser assignment

In [27]:
swaps = [
    {"Tournament": "Shanghai Masters", "Round": "1st Round", "Loser": "Yunchaokete B."},
    {"Location": "Marrakech", "Round": "2nd Round", "Winner": "Kuzmanov D."},
]

w_cols = [c for c in df_bets.columns if c.startswith('W')]
pairs = [(w, 'L' + w[1:]) for w in w_cols if ('L' + w[1:]) in df_bets.columns]

for crit in swaps:
    mask = True
    for k, v in crit.items():
        mask = mask & (df_bets[k] == v)

    for w, l in pairs:
        df_bets.loc[mask, [w, l]] = df_bets.loc[mask, [l, w]].values
    df_bets.loc[mask, ['Winner', 'Loser']] = df_bets.loc[mask, ['Loser', 'Winner']].values


In [28]:
res = compare_overlap(df_github, 'shortened_loser_name', df_bets, 'Loser')

Unique in df1[shortened_loser_name]: 694
Unique in df2[Loser]: 694
Overlap: 629

In df1[shortened_loser_name] but NOT in df2[Loser] (65):
Agustin Tirante T.
Al Mutawa J.
Alberto Olivieri G.
Alejandro Hernandez Serrano J.
Andrea Huesler M.
Aragone J.
Arnaud Bailly G.
Auger Aliassime F.
Awadhy O.
Barrios Vera T.
Bautista Agut R.
Carreno Busta P.
Cervantes Huegun I.
Chan Hong S.
Cong Mo Y.
Dolgopolov A.
Elahi Galan D.
Estrella V.
Fa Rodriguez Taverna S.
Garcia Lopez G.
Gimeno Traver D.
Gomez Gb42 A.
Haider Maurer A.
Hans Rehberg M.
Hee Lee D.
Henri Mathieu P.
Hernandez J.
Hsin Tseng C.
Hsiou Hsu Y.
Hsun Lu Y.
Hugues Herbert P.
Ignacio Londero J.
J Wolf J.
Kumar Mukund S.
Kuznetsov A.
Lennard Struff J.
Lin Wu T.
Lopez Perez E.
Manuel Cerundolo J.
Marcel Stebe C.
Marco Moroni G.
Martin Etcheverry T.
Martin del Potro J.
Meligeni Alves F.
Menendez Maceiras A.
Mingjie Lin J.
Mpetshi Perricard G.
Munoz de la Nava D.
Nicolae Madaras D.
Oconnell C.
Ortega Olmedo R.
Pablo Ficovich J.
Pablo Varilla

In [29]:
from rapidfuzz import process, fuzz

def fuzzy_suggest_unmatched(unmatched_values, candidates, *, cutoff):
    candidates = list(candidates)

    out = {}
    for v in unmatched_values:
        matches = process.extract(
            v,
            candidates,
            scorer=fuzz.token_set_ratio,
        )
        if not matches:
            continue

        if v == 'Menendez Maceiras A.':
            print(matches)
        best_score = matches[0][1]

        if best_score < cutoff:
            continue

        best_matches = [(m, s) for m, s, _ in matches if s == best_score]

        if len(best_matches) > 1:
            print(f" multiple best matches for '{v}' at score {best_score}: {[m for m, _ in best_matches]}")

        out[v] = best_matches[0]  # (match, score)

    return out

In [30]:
suggestions = fuzzy_suggest_unmatched(
    res["only_df1"],
    res["only_df2"],
    cutoff=50
)

[('Hernandez A.', 62.5, 21), ('Menendez-Maceiras A.', 55.0, 37), ('Tirante T.A.', 50.0, 57), ('Madaras D.', 46.666666666666664, 34), ('Pucinelli de Almeida M.', 46.51162790697674, 48)]


In [31]:
suggestions

{'Agustin Tirante T.': ('Tirante T.A.', 73.6842105263158),
 'Al Mutawa J.': ('Ali Mutawa J.M.', 88.88888888888889),
 'Alberto Olivieri G.': ('Olivieri G.', 100.0),
 'Alejandro Hernandez Serrano J.': ('Hernandez A.', 85.71428571428571),
 'Andrea Huesler M.': ('Huesler M.A.', 73.6842105263158),
 'Aragone J.': ('Aragone J.C.', 90.9090909090909),
 'Arnaud Bailly G.': ('Bailly G.', 100.0),
 'Auger Aliassime F.': ('Auger-Aliassime F.', 66.66666666666666),
 'Awadhy O.': ('Alawadhi O.', 80.0),
 'Barrios Vera T.': ('Barrios Vera M.T.', 93.75),
 'Bautista Agut R.': ('Bautista R.', 100.0),
 'Carreno Busta P.': ('Carreno-Busta P.', 62.5),
 'Cervantes Huegun I.': ('Cervantes I.', 100.0),
 'Chan Hong S.': ('Hong S.', 100.0),
 'Cong Mo Y.': ('Mo Y.', 100.0),
 'Dolgopolov A.': ('Dolgopolov O.', 92.3076923076923),
 'Elahi Galan D.': ('Galan D.E.', 75.0),
 'Estrella V.': ('Estrella Burgos V.', 100.0),
 'Fa Rodriguez Taverna S.': ('Rodriguez Taverna S.', 100.0),
 'Garcia Lopez G.': ('Garcia-Lopez G.', 93

In [32]:
def warn_on_duplicate_matches(suggestions):
    reverse = defaultdict(list)

    for src, (match, score) in suggestions.items():
        reverse[match].append((src, score))

    for match, sources in reverse.items():
        if len(sources) > 1:
            print(f"multiple sources mapped to '{match}':")
            for src, score in sources:
                print(f"  - {src} (score={score})")

warn_on_duplicate_matches(suggestions)

multiple sources mapped to 'Hernandez A.':
  - Alejandro Hernandez Serrano J. (score=85.71428571428571)
  - Hernandez J. (score=91.66666666666667)
  - Menendez Maceiras A. (score=62.5)


In [33]:
def replace_shortened_names(df):
    replacements = {
        "Alejandro Hernandez Serrano J.": "Hernandez A.",
        "Menendez Maceiras A.":"Menendez-Maceiras A.",
        "Hernandez J.":"Hernandez-Fernandez J.",
        'Samper Montana J.': 'Samper-Montana J.'

    }
    for old_name, new_name in replacements.items():
        df['shortened_winner_name'] = df['shortened_winner_name'].replace({old_name: new_name})
        df['shortened_loser_name'] = df['shortened_loser_name'].replace({old_name: new_name})
replace_shortened_names(df_github)

In [34]:
rename_map = {
    src: match
    for src, (match, score) in suggestions.items()
}

df_github['shortened_winner_name'] = df_github['shortened_winner_name'].replace(rename_map)
df_github['shortened_loser_name'] = df_github['shortened_loser_name'].replace(rename_map)

In [35]:
res = compare_overlap(df_github, 'shortened_loser_name', df_bets, 'Loser')

Unique in df1[shortened_loser_name]: 694
Unique in df2[Loser]: 694
Overlap: 694

In df1[shortened_loser_name] but NOT in df2[Loser] (0):

In df2[Loser] but NOT in df1[shortened_loser_name] (0):


In [36]:
res = compare_overlap(df_github, 'shortened_winner_name', df_bets, 'Winner')

Unique in df1[shortened_winner_name]: 491
Unique in df2[Winner]: 491
Overlap: 491

In df1[shortened_winner_name] but NOT in df2[Winner] (0):

In df2[Winner] but NOT in df1[shortened_winner_name] (0):


## Now we can assign player_id to df_bets

In [37]:
map_player_name_player_id = df_github.groupby('shortened_loser_name', as_index=False)[['shortened_loser_name', 'loser_id']].first()
mapping = dict(zip(map_player_name_player_id['shortened_loser_name'], map_player_name_player_id['loser_id']))

df_bets['loser_id'] = df_bets['Loser'].map(mapping)
df_bets['winner_id'] = df_bets['Winner'].map(mapping)

In [38]:
df_bets[['loser_id', 'winner_id']].isna().sum()

loser_id     0
winner_id    0
dtype: int64

## So now we can add match_id to both df

In [39]:
df_github['match_id'] = df_github['tourney_location'].astype(str) + '_' + df_github['year'].astype(str) + '_' + df_github['winner_id'].astype(str) + '_' + df_github['loser_id'].astype(str)
df_bets['match_id'] = df_bets['Location'].astype(str) + '_' + df_bets['year'].astype(str) + '_' + df_bets['winner_id'].astype(str) + '_' + df_bets['loser_id'].astype(str)

In [40]:
df_github['match_id'].is_unique

True

In [41]:
df_bets['match_id'].is_unique

True

In [42]:
merge_safe = (
    df_github['match_id'].is_unique
    and df_bets['match_id'].is_unique
    and set(df_github['match_id']) == set(df_bets['match_id'])
)
print(merge_safe)

True


In [43]:
matches = pd.merge(df_github, df_bets[['match_id', 'Date', 'Series', 'Court', 'W1', 'W2', 'W3', 'W4', 'W5', 'L1', 'L2', 'L3', 'L4', 'L5', 'Wsets', 'Lsets', 'Comment', 'AvgW', 'AvgL']], on='match_id', how='inner')

In [45]:
matches.columns

Index(['tourney_id', 'tourney_location', 'surface', 'draw_size',
       'tourney_level', 'tourney_date', 'match_num', 'winner_id',
       'winner_seed', 'winner_entry', 'winner_name', 'winner_hand',
       'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed',
       'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc',
       'loser_age', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df',
       'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved',
       'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon',
       'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'winner_rank',
       'winner_rank_points', 'loser_rank', 'loser_rank_points', 'year',
       'shortened_winner_name', 'shortened_loser_name', 'match_id', 'Date',
       'Series', 'Court', 'W1', 'W2', 'W3', 'W4', 'W5', 'L1', 'L2', 'L3', 'L4',
       'L5', 'Wsets', 'Lsets', 'Comment', 'AvgW', 'AvgL'],
      dtype='object')

In [49]:
matches.to_csv("../data/atp_matches_merged.csv", index=False)